# 3. Navigate 1910 Clusters

Inputs:
- 1910_clusters.json
- 1910_embeddings.txt
- 1910_embeddings.npy
- 1910_photos/ (jpgs)

Goals:
- Pick cluster ID -> view image grid
- Pick image -> show clusters, filename, embedding

## Setup

In [2]:
from pathlib import Path
import json
import random

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import ipywidgets as widgets
from IPython.display import display, clear_output

In [7]:
# Paths
CLUSTERS_JSON = Path("1910_clusters.json")
EMB_TXT = Path("1910_embeddings.txt")
EMB_NPY = Path("1910_embeddings.npy")
PHOTOS_DIR = Path("1910_photos")
print("Using PHOTOS_DIR:", PHOTOS_DIR)

# load clusters
clusters = json.loads(CLUSTERS_JSON.read_text(encoding="utf-8"))
print("Loaded clusters:", len(clusters))

# filenames list
filenames_1910 = [ln.strip() for ln in EMB_TXT.read_text(encoding="utf-8").splitlines() if ln.strip()]
embeddings_1910 = np.load(EMB_NPY)

print("filenames:", len(filenames_1910))
print("embeddings:", embeddings_1910.shape)

# usual alignment check
if embeddings_1910.ndim != 2:
    raise ValueError(f"Expected 2D embedding matrix, got shape={embeddings_1910.shape}")
if len(filenames_1910) != embeddings_1910.shape[0]:
    raise ValueError(f"Row mismatch: {len(filenames_1910)} filenames vs {embeddings_1910.shape[0]} embedding rows")
print("Alignment check passed.")

Using PHOTOS_DIR: 1910_photos
Loaded clusters: 2321
filenames: 72062
embeddings: (72062, 512)
Alignment check passed.


In [ ]:
# check dupes and build index <-> filename
idx_to_name = filenames_1910
name_to_idx = {}
dupes = []

for i, name in enumerate(idx_to_name):
    if name in name_to_idx:
        dupes.append(name)
    else:
        name_to_idx[name] = i

print("#unique names:", len(name_to_idx))
print("Duplicate filenames:", len(dupes))

#unique names: 72062
Duplicate filenames: 0


In [12]:
# relative jpg path -> cluster_id
jpg_to_cluster = {}

for cid, jpg_list in clusters.items():
    for jpg in jpg_list:
        # If a jpg appears twice across clusters (shouldn't), last wins
        jpg_to_cluster[jpg] = cid

# all non-noise images
print("jpg_to_cluster entries:", len(jpg_to_cluster))

jpg_to_cluster entries: 6121


## Helper utils + grid rendering

In [24]:
def jpg_exists(rel_jpg: str) -> bool:
    return (PHOTOS_DIR / rel_jpg).exists()

def open_image(rel_jpg: str) -> Image.Image:
    p = PHOTOS_DIR / rel_jpg
    if not p.exists():
        raise FileNotFoundError(f"Missing image: {p}")
    return Image.open(p).convert("RGB")

def embedding_preview_by_index(i: int, head: int = 8):
    v = embeddings_1910[i]
    return {
        "dim": int(v.shape[0]),
        "l2_norm": float(np.linalg.norm(v)),
        "first_vals": [float(x) for x in v[:head]],
    }

def filename_to_reljpg(name: str) -> str:
    stem = Path(name).stem
    return f"{stem}.jpg"

def reljpg_to_filename(rel_jpg: str) -> str:
    return str(Path(rel_jpg).with_suffix(""))  # removes .jpg

In [25]:
def show_image_grid(rel_jpgs, title="", cols=6, figsize_per_cell=2.0, max_images=60):
    rel_jpgs = list(rel_jpgs)[:max_images]
    n = len(rel_jpgs)
    if n == 0:
        print("No images to display.")
        return

    rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols * figsize_per_cell, rows * figsize_per_cell))
    for j, rel in enumerate(rel_jpgs):
        ax = plt.subplot(rows, cols, j + 1)
        try:
            img = open_image(rel)
            ax.imshow(img)
            ax.set_title(f"{j}", fontsize=8)
        except Exception as e:
            ax.text(0.5, 0.5, "ERR", ha="center", va="center")
        ax.axis("off")

    if title:
        plt.suptitle(title)
    plt.show()

## Interactive cluster browser

In [ ]:
# largest cluster appears first in the dropdown
cluster_ids_sorted = sorted(clusters.keys(), key=lambda k: len(clusters[k]), reverse=True)
max_cluster_size = max(len(clusters[c]) for c in cluster_ids_sorted) if cluster_ids_sorted else 0

# cluster UI controls
cluster_dd = widgets.Dropdown(
    options=cluster_ids_sorted,
    value=cluster_ids_sorted[0] if cluster_ids_sorted else None,
    description="Cluster:",
    layout=widgets.Layout(width="300px")
)

# Sliders
cols_slider = widgets.IntSlider(value=6, min=2, max=10, step=1, description="Cols", continuous_update=False)
max_show = widgets.IntSlider(value=60, min=12, max=240, step=12, description="Max show", continuous_update=False)
sample_n = widgets.IntSlider(value=30, min=6, max=200, step=2, description="Sample n", continuous_update=False)

# Buttons
btn_show_all = widgets.Button(description="Show (first N)", button_style="info")
btn_sample = widgets.Button(description="Sample", button_style="success")

out = widgets.Output()

# compute statistics + display grid
def render_cluster(rel_list, mode="first"):
    cid = cluster_dd.value
    if cid is None:
        print("No clusters loaded.")
        return

    rel_list = list(rel_list)
    size = len(rel_list)
    
    # mode for either first N or Sample
    title = f"Cluster {cid} | size={size} | mode={mode}"
    show_image_grid(
        rel_list,
        title=title,
        cols=cols_slider.value,
        max_images=max_show.value
    )

def on_show_all(_):
    with out:
        clear_output()
        cid = cluster_dd.value
        rel_list = clusters[cid]
        render_cluster(rel_list, mode="first")

def on_sample(_):
    with out:
        clear_output()
        cid = cluster_dd.value
        rel_list = clusters[cid]
        k = min(sample_n.value, len(rel_list))
        sampled = random.sample(rel_list, k) if k > 0 else []
        render_cluster(sampled, mode="sample")

btn_show_all.on_click(on_show_all)
btn_sample.on_click(on_sample)

display(widgets.HBox([cluster_dd, btn_show_all, btn_sample]),
        widgets.HBox([cols_slider, max_show, sample_n]),
        out)

# Auto-render first cluster
if cluster_ids_sorted:
    on_show_all(None)


Output()

## Image -> (cluster + filename + embedding)

In [31]:
# Supports with or without extension
img_text = widgets.Textarea(
    value="",
    placeholder="Paste filename from txt (no .jpg) OR a rel .jpg path from clusters.json",
    description="Image:",
    layout=widgets.Layout(width="820px", height="80px")
)

btn_inspect = widgets.Button(description="Inspect", button_style="primary")
out2 = widgets.Output()

def on_inspect(_):
    with out2:
        clear_output()
        s = img_text.value.strip()
        if not s:
            print("Paste an image id / filename / rel jpg path first.")
            return

        if s.lower().endswith(".jpg"):
            rel_jpg = s
            filename = reljpg_to_filename(rel_jpg)
        else:
            filename = s
            rel_jpg = filename_to_reljpg(filename)

        # cluster lookup
        cid = jpg_to_cluster.get(rel_jpg, None)

        # index + embedding lookup
        idx = name_to_idx.get(filename, None)

        print("Filename (txt):", filename)
        print("Rel JPG:", rel_jpg)
        print("Cluster ID:", cid if cid is not None else "Not found (maybe noise or path mismatch)")
        print("Index:", idx if idx is not None else "Not found in filenames_1910")

        if idx is not None:
            prev = embedding_preview_by_index(idx)
            print("\nEmbedding preview:")
            print("  dim:", prev["dim"])
            print("  l2_norm:", f'{prev["l2_norm"]:.4f}')
            print("  first_vals:", prev["first_vals"])

        # show image
        p = PHOTOS_DIR / rel_jpg
        if p.exists():
            img = open_image(rel_jpg)
            plt.figure(figsize=(5,5))
            plt.imshow(img)
            plt.axis("off")
            plt.title(f"cluster={cid} | idx={idx}")
            plt.show()
        else:
            print("\nImage file missing on disk:", p)

btn_inspect.on_click(on_inspect)

display(widgets.HBox([img_text, btn_inspect]), out2)

Output()